In [21]:
from sklearn.model_selection import train_test_split
import pandas as pd


In [22]:
file_path = "../data/processed/used_cars_dataset_cleaned.csv"
df = pd.read_csv(file_path)


In [23]:
# --- දත්ත පිරිසිදු කිරීමේ අමතර පියවර ---

# 1. AskPrice එකේ ඇති '₹' සහ කොමා (',') ඉවත් කර float බවට පත් කිරීම
df['AskPrice'] = df['AskPrice'].str.replace('₹', '', regex=False)
df['AskPrice'] = df['AskPrice'].str.replace(',', '', regex=False)
df['AskPrice'] = df['AskPrice'].astype(float) # ඉලක්කමක් කිරීම

# 2. kmDriven එකේ ඇති 'km', හිස්තැන් සහ කොමා (',') ඉවත් කර float බවට පත් කිරීම
# (මෙහි null values ඇති නිසා float කිරීම වඩා ආරක්ෂිතයි)
df['kmDriven'] = df['kmDriven'].str.replace('km', '', regex=False)
df['kmDriven'] = df['kmDriven'].str.replace(',', '', regex=False)
df['kmDriven'] = df['kmDriven'].str.replace(' ', '', regex=False)
df['kmDriven'] = df['kmDriven'].astype(float) # ඉලක්කමක් කිරීම

# ----------------------------------------------

In [24]:
y = df['AskPrice']
columns_to_drop = ['AskPrice','PostedDate', 'AdditionInfo', 'Age']
X = df.drop(columns_to_drop,axis=1, errors='ignore')

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['str']).columns.tolist()

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

Numeric Columns: ['kmDriven', 'Car_Age']
Categorical Columns: ['Brand', 'model', 'Transmission', 'Owner', 'FuelType']


## The Preprocessor

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [29]:
# 1. Numeric පාර (ඉලක්කම් සඳහා)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # අගයන් නැතිවූ ස්ථාන සඳහා මැද අගය භාවිතා කිරීම
    ('scaler', StandardScaler())  # දත්ත සාමාන්‍ය කිරීම
])

In [30]:
# 2. Categorical පාර (පෙළ සඳහා)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # වැඩිපුරම තියෙන අගය ඉංග්‍රීසියෙන්
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # අලුත් කාණ්ඩ සඳහා අගයන් නොමැතිවීම
])

In [31]:
# 3. පාරවල් දෙක එකතු කිරීම (The Grand Join)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),  # ඉලක්කම් සඳහා පාර
        ('cat', categorical_transformer, categorical_cols)  # පෙළ සඳහා පාර
    ])

In [32]:
import joblib
import os

# 1. Models සහ Processed Data ෆෝල්ඩර් නැත්නම් හදන්න
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# 2. Preprocessor එක (ML Pipeline කොටස) Save කිරීම
joblib.dump(preprocessor, '../models/preprocessor.pkl')

# 3. Train සහ Test Data ටික Save කිරීම (CSV විදිහට)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Preprocessor සහ Data සාර්ථකව Save කරගත්තා!")

Preprocessor සහ Data සාර්ථකව Save කරගත්තා!
